In [180]:
import itk
import re
import numpy as np
import os
import matplotlib.pyplot as plt
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

import seaborn as sns
from IPython.display import display
import sys 
import numpy as np 
import pandas as pd 
import matplotlib as mpl
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from matplotlib.pyplot import plot, ion, show
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.path import Path
# from matplotlib import pyplot as plt
# sns.set_theme(context='notebook',style='ticks',palette=None)
# sns.despine()


In [ ]:
ROOT = r"D:\MDH_DATA\09_04_2025\Patient_1\ThermalData\Cropped"

reference_img = "Snap-000143.npz"

hot_video = "Rec-000145_registered.npz"
cold_video = 'Rec-000146_registered.npz'

hyp_image = '2025-04-09_001.png'




# Referece Thermal Image
ref_img = np.load(os.path.join(ROOT,reference_img))['reg_image']

# Hyperspectral Image
hyp_img = cv2.imread(os.path.join(ROOT,hyp_image))


In [ ]:


def mark_coordinates_on_image(img):
    """
    Display an image and allow marking coordinates with clicks.
    Saves the image with markers when closed.
    
    Args:
        image_path (str): Path to input image
        output_path (str): Path to save marked image
    Returns:
        list: Clicked coordinates [(x1,y1), (x2,y2),...]
    """
   
    fig, ax = plt.subplots()
    ax.imshow(img,cmap='viridis')
    clicked_coords = []
    
    def onclick(event):
        """Handle mouse clicks and mark points"""
        if event.inaxes == ax:
            x, y = event.xdata, event.ydata
            clicked_coords.append((x, y))
            print(f"Marked point at: ({x:.1f}, {y:.1f})")
            
            # Plot red cross and force immediate render

            ax.plot(x, y, 'r+', markersize=10, markeredgewidth=2)
            fig.canvas.draw_idle()  # Update display immediately

    
    # Connect events
    fig.canvas.mpl_connect('button_press_event', onclick)
    plt.show()
    return clicked_coords,fig,ax

In [33]:
def main_plot(coords,fig,ax,name):
    
    coords = np.array(coords)
    ax.plot(coords[:,0],coords[:,1],'r+')
    ax.set_axis_off()
    fig.savefig(name,bbox_inches='tight', pad_inches=0, dpi=300)

In [43]:
%matplotlib tk

# Getting Coordinates of Hyperspectral Image to crop it
# Steps : 
# First select the top left coordinate
# Second Selct the bottom right cooridate
# Alwasy ensure to capture all of the markers into one image
cropped_coords,fig,ax= mark_coordinates_on_image(hyp_img[:,:,::-1])





In [44]:
main_plot(cropped_coords,fig,ax,'HyperspectralImage.png')

In [45]:
cropped_hyp_image = hyp_img[int(cropped_coords[0][1]):int(cropped_coords[1][1]),int(cropped_coords[0][0]):int(cropped_coords[1][0])]

In [46]:
plt.imshow(cropped_hyp_image)

In [47]:
# %matplotlib tk

# Getting Coordinates of Markers of Hyperspectral Camera

ref_coords,fig,ax = mark_coordinates_on_image(cropped_hyp_image[:,:,::-1])

In [48]:
main_plot(ref_coords,fig,ax,'MarkersLocationHyperspectralImage.png')

In [ ]:
# If number of markers are higher than 4 then this needs to be adjusted accordingly
assert len(ref_coords) == 3,"3 markers should be selected"

In [50]:

# Getting Coordinates of Thermal Image First Frame
mov_coords,fig,ax = mark_coordinates_on_image(ref_img)


In [51]:
main_plot(mov_coords,fig,ax,'MarkerLocationofThermalImage.png')

In [52]:
assert len(mov_coords) == 3,"3 markers should be selected"

In [ ]:
def register_frames(ref_img,moving_image,
                    fixed_coords,
                    moving_coords):
    fixed_image = itk.GetImageFromArray(ref_img)
    mvi_image = itk.GetImageFromArray(moving_image)

    """
    This function is used to register two frames based on the coordinates
    
    """


    mov_points = np.array([list(map(float,i[:2])) for i in moving_coords]).tolist()
    ref_ori = np.array([list(map(float,i[:2])) for i in fixed_coords]).tolist()


    LandMarkPoinType = itk.Point[itk.D,2]
    LandMarkContainerType = itk.vector[LandMarkPoinType]

    fixed_landmarks = LandMarkContainerType()
    fixed_point = LandMarkPoinType()

    moving_landmarks = LandMarkContainerType()

    moving_point = LandMarkPoinType()


    for x in ref_ori:
        fixed_point[0] = x[0]
        fixed_point[1] = x[1]
        fixed_landmarks.push_back(fixed_point)


    for x in mov_points:
            moving_point[0] = x[0]
            moving_point [1] = x[1]
            moving_landmarks.push_back(moving_point)


    TransformInitializerType = itk.LandmarkBasedTransformInitializer[
    itk.Transform[itk.D,2,2]]

    transform_initializer = TransformInitializerType.New()

    transform_initializer.SetFixedLandmarks(fixed_landmarks)
    transform_initializer.SetMovingLandmarks(moving_landmarks)

    transform = itk.AffineTransform[itk.D,2].New()

    # transform = itk.Rigid2DTransform[itk.D].New()
    transform_initializer.SetTransform(transform)
    transform_initializer.InitializeTransform()


    output = itk.resample_image_filter(
    mvi_image,
    transform=transform,
    use_reference_image=True,
    reference_image = fixed_image,
    default_pixel_value = 0
    )


    final = itk.GetArrayFromImage(output)


    source = itk.GetArrayFromImage(fixed_image)

    # return output,mvi_image,transform
    return final,source,transform,output





    
   




    
    


In [54]:
cropped_hyp_image = cv2.cvtColor(cropped_hyp_image,cv2.COLOR_BGR2GRAY)

In [55]:
hyp_img = cv2.imread(os.path.join(ROOT,hyp_image),cv2.IMREAD_GRAYSCALE)
# Hyperspctral Image is Fixed --> Reference Frame will be transformed into a way of Hypespectral Image
transformed_image_ref,fixed_image,transform,output = register_frames(cropped_hyp_image,ref_img,ref_coords,mov_coords)

In [56]:


# Create figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

# Display the images
ax1.imshow(transformed_image_ref , cmap='gray')
ax2.imshow(fixed_image, cmap='gray')
ax1.set_title('Click on this image')
ax2.set_title('Corresponding point will appear here')

# Initialize a list to store the marked points
marked_points = []

def onclick(event):
    # Only respond to clicks in the first axis
    if event.inaxes != ax1:
        return
    
    # Clear previous marks
    for mark in marked_points:
        mark.remove()
    marked_points.clear()
    
    # Get click coordinates
    x, y = event.xdata, event.ydata
    
    # Mark the clicked point on first image with a red circle
    mark1 = ax1.plot(x, y, 'ro', markersize=8, alpha=0.7)[0]
    
    # Mark the corresponding point on second image
    mark2 = ax2.plot(x, y, 'ro', markersize=8, alpha=0.7)[0]
    
    # Add text annotations
    text1 = ax1.text(x, y + 5, f'({x:.1f}, {y:.1f})', color='red')
    text2 = ax2.text(x, y + 5, f'({x:.1f}, {y:.1f})', color='red')
    
    # Store references to the marks
    marked_points.extend([mark1, mark2])
    
    # Redraw the figure
    fig.canvas.draw()

# Connect the click event to the function
fig.canvas.mpl_connect('button_press_event', onclick)

plt.tight_layout()
plt.show()


plt.savefig('transformation_hypespectral_to_thermal_image.png')  

Thermal Stimulation on Hypespectral Image

In [ ]:
# #  Artificial Generated Image in SF12
# artificial_generated_img = r'C:\Users\nipun\Videos\SF12\AneAne.png'
# arti_img = cv2.imread(artificial_generated_img)
# # # Resise image as thermal frame
# arti_img = cv2.resize(arti_img,(hot_first_frame.shape[1],hot_first_frame.shape[0]))

# artificial_coordinates = mark_coordinates_on_image(arti_img)


In [ ]:
# hot_first_frame = cv2.imread(os.path.join(ROOT,'Hot_Fame.png'))
# hot_first_frame = cv2.resize(hot_first_frame,(481,301))
# hot_first_frame= cv2.cvtColor(hot_first_frame,cv2.COLOR_BGR2GRAY)

In [ ]:
# cold_first_frame = cv2.imread(os.path.join(ROOT,'Cold_Frame.png'))
# cold_first_frame = cv2.resize(cold_first_frame,(405,259))
# cold_first_frame = cv2.cvtColor(cold_first_frame,cv2.COLOR_BGR2GRAY)

In [57]:
# Take the First frame of both thermal videos
hot_first_frame = np.load(os.path.join(ROOT,hot_video))['cropped_vid'][:,:,0] 
cold_first_frame = np.load(os.path.join(ROOT,cold_video))['cropped_vid'][:,:,0] 


# hot_first_frame = np.load('hot_array.npy')
# cold_first_frame = np.load('cold_array.npy')

In [58]:
# Getting Coordinates of Thermal Image First Frame
hot_coords,fig,ax = mark_coordinates_on_image(hot_first_frame)

# only True in SF12
# hot_coords = artificial_coordinates[0]


In [59]:
cold_coords,fig,ax = mark_coordinates_on_image(cold_first_frame)


In [60]:
# Hyperspctral Image is Fixed --> hot Frame will be transformed into a way of Hypespectral Image
hot_transformed_image,fixed_image,transform,output = register_frames(cropped_hyp_image,hot_first_frame,ref_coords,hot_coords)

In [ ]:


# Create figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

# Display the images
ax1.imshow(hot_transformed_image , cmap='gray')
ax2.imshow(fixed_image, cmap='gray')
ax1.set_title('Click on this image')
ax2.set_title('Corresponding point will appear here')
# Connect the click event to the function
fig.canvas.mpl_connect('button_press_event', onclick)
plt.tight_layout()
plt.show()
plt.savefig('transformation_hypespectral_to_hot_thermal_image.png')  

In [62]:
cold_transformed_image,fixed_image,transform,output = register_frames(cropped_hyp_image,cold_first_frame,ref_coords,cold_coords)

In [64]:


# Create figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

# Display the images
ax1.imshow(cold_transformed_image , cmap='gray')
ax2.imshow(fixed_image, cmap='gray')
ax1.set_title('Click on this image')
ax2.set_title('Corresponding point will appear here')
fig.canvas.mpl_connect('button_press_event', onclick)
plt.tight_layout()
plt.show()
plt.savefig('transformation_hypespectral_to_cold_thermal_image.png')  

In [65]:
# Obtaining Hot thermal Stimulation on Transformed Image
hot_transformed_location,fig,ax = mark_coordinates_on_image(hot_transformed_image)

In [66]:
# Obtaining Cold thermal Stimulation on Transformed Image

cold_transformed_location,fig,ax = mark_coordinates_on_image(cold_transformed_image)

In [67]:
rgb_hyp = cv2.imread(os.path.join(ROOT,hyp_image))[int(cropped_coords[0][1]):int(cropped_coords[1][1]),int(cropped_coords[0][0]):int(cropped_coords[1][0])]

In [68]:
hot_stimulation_area = np.array(hot_transformed_location,np.int32).reshape(-1,1,2)

cold_stimulation_area = np.array(cold_transformed_location,np.int32).reshape(-1,1,2)


In [69]:
hot_stimulation_area.shape

(8, 1, 2)

In [70]:
# np.save('hot_stimulation_area.npz',hot_stimulation_area)
# np.save('cold_stimulation_area.npz',cold_stimulation_area)

In [71]:
# hot_stimulation_area = np.load('hot_stimulation_area.npz.npy')
# cold_stimulation_area = np.load('cold_stimulation_area.npz.npy')

In [72]:
cv2.polylines(rgb_hyp,[hot_stimulation_area],isClosed=True,color=(255,0,0),thickness=1) # Blue Hot
cv2.polylines(rgb_hyp,[cold_stimulation_area],isClosed=True,color=(0,255,0),thickness=1) # Green Cold



array([[[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [210, 227, 223],
        [211, 226, 220],
        [216, 230, 223]],

       [[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [210, 227, 222],
        [211, 228, 223],
        [218, 233, 228]],

       [[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [215, 228, 225],
        [216, 232, 226],
        [220, 233, 229]],

       ...,

       [[ 51,  72, 111],
        [ 52,  73, 112],
        [ 54,  75, 115],
        ...,
        [182, 191, 192],
        [185, 193, 192],
        [189, 197, 196]],

       [[ 49,  71, 110],
        [ 51,  72, 111],
        [ 54,  73, 112],
        ...,
        [185, 193, 193],
        [176, 187, 187],
        [178, 185, 187]],

       [[ 51,  74, 112],
        [ 53,  75, 113],
        [ 53,  75, 114],
        ...,
        [199, 205, 205],
        [187, 194, 195],
        [180, 187, 190]]

In [73]:
plt.imshow(rgb_hyp[:,:,::-1])

In [74]:
%matplotlib tk



# Getting Coordinates of  lesion Thermal Image First Frame
lesion_locations,fig,ax = mark_coordinates_on_image(rgb_hyp[:,:,::-1])


In [76]:
skin_locations,fig,ax = mark_coordinates_on_image(rgb_hyp[:,:,::-1])

In [77]:
lesion_area = np.array(lesion_locations,dtype=np.int32).reshape(-1,1,2)
skin_area = np.array(skin_locations,dtype=np.int32).reshape(-1,1,2)

In [78]:
cv2.polylines(rgb_hyp,[lesion_area],isClosed=True,color=(0,0,255),thickness=1)
cv2.polylines(rgb_hyp,[skin_area] ,isClosed=True,color=(0,0,255),thickness=1)

array([[[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [210, 227, 223],
        [211, 226, 220],
        [216, 230, 223]],

       [[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [210, 227, 222],
        [211, 228, 223],
        [218, 233, 228]],

       [[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [215, 228, 225],
        [216, 232, 226],
        [220, 233, 229]],

       ...,

       [[ 51,  72, 111],
        [ 52,  73, 112],
        [ 54,  75, 115],
        ...,
        [182, 191, 192],
        [185, 193, 192],
        [189, 197, 196]],

       [[ 49,  71, 110],
        [ 51,  72, 111],
        [ 54,  73, 112],
        ...,
        [185, 193, 193],
        [176, 187, 187],
        [178, 185, 187]],

       [[ 51,  74, 112],
        [ 53,  75, 113],
        [ 53,  75, 114],
        ...,
        [199, 205, 205],
        [187, 194, 195],
        [180, 187, 190]]

In [79]:
plt.imshow(rgb_hyp[:,:,::-1])

In [80]:
cv2.imwrite('MarkedHyperspectralImage.png',rgb_hyp)

True

In [81]:
UNCHANGED_SKIN = skin_locations
UNCHANGED_LESION = lesion_locations

In [82]:
# main_plot(skin_locations,fig,ax,'skin_locations.png')


# Transform Back

In [ ]:
# Important : Always check what kind of video is being referred:

video_type ='hot'

if video_type =='hot':
    video = hot_video
    plot_type= 'max'
else:
    video= cold_video
    plot_type='min'


#  Load the thermal Video :
vid = np.load(os.path.join(ROOT,video))['cropped_vid']

# ref_img = vid[:,:,0]
# Take the first image of thermal image as reference image 
move_image = vid[:,:,0]



In [ ]:
# Extract the simulation area:
# Task: You have to mark the simulation area : Most of the time the shape of this looks like a rectangle
# Important : Please carefully analyse the Thermal Stimulation sometimes thermal stimulation may show irregular shape
stimulation_area,fig,ax = mark_coordinates_on_image(move_image)

In [ ]:
# Getting Coordinates First Frame of Thermal Video
# Task: Mark the center of each reflective marker
ref_up_coords,fig,ax  = mark_coordinates_on_image(move_image)


In [86]:
main_plot(ref_up_coords,fig,ax,'back_transformation_refereceimage.png')

In [ ]:
%matplotlib tk

# Getting Coordinates of transformed therma camera first frame
# Task : Mark the center of each reflective marker

"""
We use the thermal reference image because, during data acquisition, we first captured the hyperspectral image followed by the thermal reference image.

Since patient movement between these two images is relatively small compared to movement in the thermal video, the images are better aligned. Therefore, we can use the thermal reference image, which is already aligned with the hyperspectral image, as an intermediate step to align with the thermal video.

Using this alignment, we can compute a transformation matrix that allows us to approximate the locations of lesions and healthy skin in the thermally stimulated video.

"""

transfromed_move_coords,fig,ax = mark_coordinates_on_image(transformed_image_ref)



In [88]:
main_plot(ref_up_coords,fig,ax,'transformed_image.png')

In [ ]:
# Register the reference image (captured prior to thermal stimulation and already aligned with the hyperspectral image) with the first frame of the thermal video.

# In this way, we do not need to register every frame of the thermal video again.
transformed_image,fixed_image,transform,output = register_frames(move_image,
                                                          transformed_image_ref,
                                                          ref_up_coords,
                                                          transfromed_move_coords)

In [145]:


# Create figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

# Display the images
ax1.imshow(fixed_image , cmap='gray')
ax2.imshow(transformed_image, cmap='gray')
ax1.set_title('Click on this image')
ax2.set_title('Corresponding point will appear here')
# Connect the click event to the function
fig.canvas.mpl_connect('button_press_event', onclick)

plt.tight_layout()
plt.show()
plt.savefig('backtransformation_transformedfirstthermalimageto_originalfirstthermalframe.png')

In [ ]:
def transform_coords(transform,array):

    """
    Transform any given array based on Transformation Matrix
    Usage: Transform original Skin Lesion Location into the thermal video frames.
    
    """
    mapping_original_transform=transform.GetInverseTransform()

    mapped_coords = [mapping_original_transform.TransformPoint(i) for i in array]

    return [[mapped_coords  [i].GetElement(0),mapped_coords [i].GetElement(1)]   for i in range(len(mapped_coords  ))]




In [ ]:
# Transformation of Coordinates (Lesion Loc & Healthy Loc)
lesion_loc_on_original = transform_coords(transform,lesion_locations)
skin_loc_on_original = transform_coords(transform,skin_locations)

In [177]:
def drawlocations_OPENCV(image,lesion_location,stimulation_loc,skin_loc=None):
    
    """
    Simple Utility Function to Draw Small Circle on given 
    """
    

    image = image.copy()

    if skin_loc is not None:
        for elem in skin_loc:
            cv2.circle(image,(int(elem[0]),int(elem[1])),1,(255,0,0),1,-1)

    for elem in lesion_location:
            cv2.circle(image,(int(elem[0]),int(elem[1])),1,(0,255,0),1,-1)

    for elem in stimulation_loc:
            cv2.circle(image,(int(elem[0]),int(elem[1])),1,(0,255,0),1,-1)


    return image

In [ ]:
# Reference Image : This means still we consider the reference image as the image which we captured to prior thermal stimulation
# However this reference image has gone number of registration steps.
reference_img = np.array(transformed_image)

In [151]:

# Subsampling: In here we select thermal frame after every 2 second
frames_per_second =30
interval = 2

frames_interval = frames_per_second * interval

In [152]:
# Sub sampling from every 2 seconds
# 0,32,64...... -> 1,33,65(Python index which starts 0) after every 2 second first frame is obtaineds
sampled_video = vid[:,:,::frames_interval]

assert np.all(vid[:,:,frames_interval] == sampled_video[:,:,1]), 'Sub Sampling is not working properly'


In [153]:
# Reshape new_array to (H, W, Channel) to match dimensions
new_array_reshaped = reference_img[:, :, np.newaxis]

# Concatenate along the first axis (axis=0)
combined_array = np.concatenate((new_array_reshaped, sampled_video), axis=-1)

In [ ]:
# Use this code block to understand how lesion location has spreaded across the image

ref_img = combined_array[:,:,0]
img = ref_img.copy()

plt.imshow(drawlocations(img,lesion_loc_on_original,stimulation_area,skin_loc_on_original))



In [161]:
sampled_video = combined_array.copy()


# sampled_video = sampled_video[:,:,:90]

## Temperature Profile


**Idea:** Select a point on the skin lesion (usually the center). Then, draw a vertical line passing through this point and analyze how the temperature changes along this line before and after thermal stimulation.



In [164]:
%matplotlib tk

point_selection,_,_ = mark_coordinates_on_image(img)


In [165]:

point_selection[0]
plt.style.use('default')

In [167]:
#  Find Maximum X and Minimum X of  thermal stimulation
stimulation_loc = np.array(stimulation_area,np.float32)
MAX_X_STIMUL = stimulation_loc[:,0].max()
MIN_X_STIMUL = stimulation_loc[:,0].min()

In [110]:
#  Find Maximum X and Minimum X of  lesion
lesion_loc = np.array(lesion_loc_on_original,np.float32)
MAX_X_LESION = lesion_loc[:,0].max()
MIN_X_LESION = lesion_loc[:,0].min()

In [178]:
def temp_profile_analyse(ref_image, firstframe, point, min_lesion, max_lesion, min_stimul, max_stimul, image):


    fig = plt.figure(figsize=(12, 8))
    gs = fig.add_gridspec(2, 2, width_ratios=[2, 1])
    
    ax1 = fig.add_subplot(gs[:, 0])
    ax1.set_title('Temperature Profiles', fontsize=12)
    
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.set_title('Ref Image', fontsize=12)
    ax2.set_aspect('equal')
    
    ax3 = fig.add_subplot(gs[1, 1]) 
    ax3.set_title('FF Image', fontsize=12)
    ax3.set_aspect('equal')
    
    # Temperature profiles
    temperature_profile_of_referece_image = ref_image[int(point[0][1]), :]
    firstframe_temperature_profile = firstframe[int(point[0][1]), :]
    
    # Crop temperature profiles to stimulation boundaries
    min_idx = int(min_stimul)
    max_idx = int(max_stimul)
    
    cropped_ref_profile = temperature_profile_of_referece_image[min_idx:max_idx+1]
    cropped_ff_profile = firstframe_temperature_profile[min_idx:max_idx+1]
    
    # Create x-axis values for cropped data (relative to stimulation start)
    x_values = range(len(cropped_ref_profile))
    
    # Plot cropped temperature profiles
    ax1.plot(x_values, cropped_ref_profile, 
             label='Ref TP', color='green', linewidth=2)
    
    ax1.plot(x_values, cropped_ff_profile, 
             label='FF TP', color='red', linewidth=2)
    
    # Adjust lesion boundaries relative to cropped data
    lesion_min_relative = int(min_lesion) - min_idx
    lesion_max_relative = int(max_lesion) - min_idx
    
    # Only plot lesion markers if they fall within the stimulation bounds
    if 0 <= lesion_min_relative < len(cropped_ref_profile):
        ax1.plot(lesion_min_relative, cropped_ref_profile[lesion_min_relative], 
                 'o', color='yellow', markersize=8)
        ax1.plot(lesion_min_relative, cropped_ff_profile[lesion_min_relative], 
                 'o', color='orange', markersize=8)
        ax1.axvline(x=lesion_min_relative, linestyle='--', color='red', 
                   linewidth=1.5, alpha=0.5, label='Lesion bounds')
    
    if 0 <= lesion_max_relative < len(cropped_ref_profile):
        ax1.plot(lesion_max_relative, cropped_ref_profile[lesion_max_relative], 
                 'o', color='yellow', markersize=8)
        ax1.plot(lesion_max_relative, cropped_ff_profile[lesion_max_relative], 
                 'o', color='orange', markersize=8)
        ax1.axvline(x=lesion_max_relative, linestyle='--', color='red', 
                   linewidth=1.5, alpha=0.5)
    
    # Stimulation boundaries are now at the edges (0 and end)
    ax1.axvline(x=0, linestyle='--', color='green', linewidth=1.5, alpha=0.5, 
               label='Stimulation bounds')
    ax1.axvline(x=len(cropped_ref_profile)-1, linestyle='--', color='green', 
               linewidth=1.5, alpha=0.5)
    
    ax1.legend(loc='lower right', fontsize=10, framealpha=1)
    ax1.grid(True, linestyle=':', alpha=0.3)
    ax1.set_xlabel('Position (relative to stimulation start)')
    ax1.set_ylabel('Temperature')
    
    # Image plots (assuming drawlocations is defined elsewhere)
    img2 = drawlocations_OPENCV(ref_image, lesion_loc_on_original, stimulation_loc)
    img3 = drawlocations_OPENCV(move_image, lesion_loc_on_original, stimulation_loc)
    
    ax2.imshow(img2)
    ax3.imshow(img3)

    # Vertical lines on images (keeping original coordinates)
    ax2.axvline(x=min_lesion, linestyle='--', color='red', linewidth=1.5, alpha=0.5, label='Lesion bounds')
    ax2.axvline(x=max_lesion, linestyle='--', color='red', linewidth=1.5, alpha=0.5)
    ax2.axvline(x=min_stimul, linestyle='--', color='green', linewidth=1.5, alpha=0.5, label='Stimulation bounds')
    ax2.axvline(x=max_stimul, linestyle='--', color='green', linewidth=1.5, alpha=0.5)

    ax3.axvline(x=min_lesion, linestyle='--', color='red', linewidth=1.5, alpha=0.5, label='Lesion bounds')
    ax3.axvline(x=max_lesion, linestyle='--', color='red', linewidth=1.5, alpha=0.5)
    ax3.axvline(x=min_stimul, linestyle='--', color='green', linewidth=1.5, alpha=0.5, label='Stimulation bounds')
    ax3.axvline(x=max_stimul, linestyle='--', color='green', linewidth=1.5, alpha=0.5)
    
    plt.tight_layout()
    fig.savefig('TemperatureProfile.png', dpi=300, bbox_inches='tight')
    plt.show()

In [179]:
temp_profile_analyse(ref_img,move_image,point_selection,MIN_X_LESION,MAX_X_LESION,MIN_X_STIMUL,MAX_X_STIMUL,img)

In [113]:
plt.close()

In [114]:
# healthy_skin_locations

In [ ]:
def obtain_contour(lesion_location,image):

    """
    Create a mask from a polygonal region representing a lesion on an image, extract the pixels within this region, and compute the mean intensity value of those pixels.

    """

    polygan_path = Path(lesion_location)

    height,width = image.shape

    y, x = np.mgrid[:height, :width]

    pixel_coords = np.column_stack((x.ravel(), y.ravel()))

    mask = polygan_path.contains_points(pixel_coords)

    mask = mask.reshape(height,width)

    masked_image = image.copy()

    masked_image[~mask] = 0

    mean_of_contour = np.mean(masked_image[mask])

    return masked_image,mean_of_contour



In [116]:
def drawlocations(image, lesion_location, stimulation_loc, skin_loc=None):
    """
    Draw locations and polygons on an image using matplotlib
    
    Parameters:
        image: Input image (numpy array)
        lesion_location: List of lesion points [(x1,y1), (x2,y2), ...]
        stimulation_loc: List of stimulation points [(x1,y1), (x2,y2), ...]
        skin_loc: Optional list of skin points [(x1,y1), (x2,y2), ...]
    
    Returns:
        Image array with drawn points and polygons
    """
    # Create figure and axis
    fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
    fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
    
    # Display the image
    ax.imshow(image)
    
    # Define colors and styles
    colors = {
        'skin':   (0.2, 0.8, 0.2, 0.3) ,  # Blue with transparency
        'lesion': (0.8, 0.2, 0.2, 0.3),  # Green with transparency
        'stimulation': (0.2, 0.4, 0.8, 0.3)  # Red with transparency
    }
    edge_colors = {
        'skin':   (0.2, 0.8, 0.2, 0.8) ,  # Blue with transparency
        'lesion': (0.8, 0.2, 0.2, 0.),  # Green with transparency
        'stimulation': (0.2, 0.4, 0.8, 0.8)
    }
    point_size = 20
    
    # Draw skin points and polygon
    if skin_loc is not None and len(skin_loc) > 0:
        skin_points = np.array(skin_loc)
        ax.scatter(skin_points[:, 0], skin_points[:, 1], s=point_size, 
                  color=colors['skin'][:3], label='Skin')
        if len(skin_loc) >= 3:
            skin_poly = Polygon(skin_points, closed=True, 
                              facecolor=colors['skin'], 
                              edgecolor=edge_colors['skin'],
                              linewidth=1.5)
            ax.add_patch(skin_poly)
    
    # Draw lesion points and polygon
    if lesion_location is not None and len(lesion_location) > 0:
        lesion_points = np.array(lesion_location)
        ax.scatter(lesion_points[:, 0], lesion_points[:, 1], s=point_size,
                  color=colors['lesion'][:3], label='Lesion')
        if len(lesion_location) >= 3:
            lesion_poly = Polygon(lesion_points, closed=True,
                                facecolor=colors['lesion'],
                                edgecolor=edge_colors['lesion'],
                                linewidth=1.5)
            ax.add_patch(lesion_poly)
    
    # Draw stimulation points and polygon
    if stimulation_loc is not None and len(stimulation_loc) > 0:
        stim_points = np.array(stimulation_loc)
        ax.scatter(stim_points[:, 0], stim_points[:, 1], s=point_size,
                  color=colors['stimulation'][:3], label='Stimulation')
        if len(stimulation_loc) >= 3:
            stim_poly = Polygon(stim_points, closed=True,
                              facecolor=colors['stimulation'],
                              edgecolor=edge_colors['stimulation'],
                              linewidth=1.5)
            ax.add_patch(stim_poly)
    
    # Configure plot appearance
    ax.set_axis_off()
    
    # Render to numpy array
    fig.canvas.draw()
    img_array = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    img_array = img_array.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    plt.close(fig)
    
    return img_array

In [172]:
def plot_lesion_temperature(video, lesion_locations, skin_locations, thermal_stimulation_type,stimulation_loc=None):
    # Create figure with adjusted size and DPI
    fig = plt.figure(figsize=(12, 6), dpi=300)

    # Create subplots with adjusted spacing
    gs = fig.add_gridspec(1, 2, width_ratios=[1.5, 1])
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])
    
    # Set titles with larger font
    ax1.set_title('Mean Temperature Over Frames', fontsize=14, pad=20)
    ax2.set_title('Reference Frame', fontsize=14, pad=20)
    
    # Process video frames
    lesion_tmps = []
    skin_tmps = []
    for i in range(video.shape[-1]):

       

        image = video[:,:,i]

        # if i> 177:
        #         image += 1.75
        contour_lesion, mean_temp_lesion = obtain_contour(lesion_locations, image)
        skin_lesion, mean_temp_skin = obtain_contour(skin_locations, image)

        print(f'Frame Number {i}: Skin Lesion Temperature {mean_temp_lesion}')
        print(f'Frame Number {i}: Skin  Temperature {mean_temp_skin}')
        
        lesion_tmps.append(mean_temp_lesion)
        skin_tmps.append(mean_temp_skin)
        
        if i == 1:  # Save sample images
            plt.imsave('lesion_image.png', contour_lesion, dpi=300)
            plt.imsave('skin_image.png', skin_lesion, dpi=300)
    
    # Plot temperature curves with thicker lines
    line1, = ax1.plot(lesion_tmps, label='Lesion', color='#d62728', linewidth=1)
    line2, = ax1.plot(skin_tmps, label='Healthy Skin', color='#2ca02c', linewidth=1)
    
    # Calculate smart annotation positions
    y_range = max(max(lesion_tmps), max(skin_tmps)) - min(min(lesion_tmps), min(skin_tmps))
    offset_y = y_range * 0.05  # 5% of y range as vertical offset
    
    # Annotate points with non-overlapping positions
    def annotate_points(ax, values, color, position, vertical_pos=None):
        # Common annotation style
        bbox_style = dict(boxstyle='round,pad=0.1', fc='white', alpha=0.9, 
                         edgecolor=color, linewidth=1)
        
        # Different arrow styles for different positions
        if position == 'end':
            arrow_style = dict(arrowstyle="-|>", color=color, linewidth=1)  # Normal size
        else:
            arrow_style = dict(arrowstyle="-|>", color=color, linewidth=1)  # Smaller size
        
        if position == 'start':
             
            if color == '#2ca02c':  # Skin (top)
                # Add cross marker at start point
                ax.plot(0, values[0], 'o', color=color, markersize=4, markeredgewidth=1)
                
                # Both start annotations on right side
                ax.annotate(f'Start: {values[0]:.2f}°C', 
                        xy=(0, values[0]), 
                        xytext=(60, 0),
                        textcoords='offset points',
                        bbox=bbox_style,
                        fontsize=5,
                        color=color,
                        arrowprops=arrow_style,
                        ha='left',
                        va='center')
            
            else:
                      # Lesion (bottom)
                    # Add cross marker at start point
                    ax.plot(0, values[0], 'o', color=color, markersize=4, markeredgewidth=1)
                    
                    # Both start annotations on right side
                    ax.annotate(f'Start: {values[0]:.2f}°C', 
                            xy=(0, values[0]), 
                            xytext=(60, 10),
                            textcoords='offset points',
                            bbox=bbox_style,
                            fontsize=5,
                            color=color,
                            arrowprops=arrow_style,
                            ha='left',
                            va='center')
                
                    
            
        elif position == 'end':
                # Add cross marker at end point
                ax.plot(len(values)-1, values[-1], 'o', color=color, markersize=4, markeredgewidth=1)
                
                # End annotations - one above, one below
                if color == '#2ca02c':  # Skin (top)
                    ax.annotate(f'End: {values[-1]:.2f}°C', 
                            xy=(len(values)-1, values[-1]), 
                            xytext=(0, 30),
                            textcoords='offset points',
                           bbox=bbox_style,
                           fontsize=5,
                           color=color,
                           arrowprops=arrow_style,
                           ha='center',
                           va='bottom')
                else:  # Lesion (bottom)
                    ax.annotate(f'End: {values[-1]:.2f}°C', 
                           xy=(len(values)-1, values[-1]), 
                           xytext=(0, -30),
                           textcoords='offset points',
                           bbox=bbox_style,
                           fontsize=5,
                           color=color,
                           arrowprops=arrow_style,
                           ha='center',
                           va='top')
                
        elif position == 'max':
            max_idx = np.argmax(values)
            max_val = max(values)
            
            if color == '#2ca02c':  # Skin (top)
                ax.plot(1, values[1], 'o', color=color, markersize=4, markeredgewidth=1)
                ax.annotate(f'Max: { values[1]:.2f}°C', 
                        xy=(1, values[1]), 
                        xytext=(60, 0),
                        textcoords='offset points',
                        bbox=bbox_style,
                        fontsize=5,
                        color=color,
                        arrowprops=arrow_style,
                        ha='left',
                        va='center')
            else:  # Lesion (bottom)
                ax.plot(1, values[1], 'o', color=color, markersize=4, markeredgewidth=1)
                ax.annotate(f'Max: { values[1]:.2f}°C', 
                        xy=(1, values[1]), 
                        xytext=(60, 15),
                        textcoords='offset points',
                        bbox=bbox_style,
                        fontsize=5,
                        color=color,
                        arrowprops=arrow_style,
                        ha='left',
                        va='center')
            
        elif position == 'min':
            min_idx = 0
            min_val = values[1]

            # End annotations - one above, one below
            if color == '#2ca02c':  # Skin (top)
                ax.plot(1, min_val, 'o', color=color, markersize=4, markeredgewidth=1)
                ax.annotate(f'Min: {min_val:.2f}°C', 
                        xy=(min_idx, min_val), 
                        xytext=(60, 0),
                        textcoords='offset points',
                        bbox=bbox_style,
                        fontsize=5,
                        color=color,
                        arrowprops=arrow_style,
                        ha='left',
                        va='center')
            else:  # Lesion (bottom)
                ax.plot(min_idx, min_val, 'o', color=color, markersize=4, markeredgewidth=1)
                ax.annotate(f'Min: {min_val:.2f}°C', 
                        xy=(min_idx, min_val), 
                        xytext=(60, 15),
                        textcoords='offset points',
                        bbox=bbox_style,
                        fontsize=5,
                        color=color,
                        arrowprops=arrow_style,
                        ha='left',
                        va='center')

            # Add cross marker at max point
            

    # Apply annotations
    annotate_points(ax1, lesion_tmps, "#d62728", 'start')
    annotate_points(ax1, skin_tmps, '#2ca02c', 'start')
    annotate_points(ax1, lesion_tmps, '#d62728', 'end')
    annotate_points(ax1, skin_tmps, '#2ca02c', 'end')
    if thermal_stimulation_type=='max':
        annotate_points(ax1, lesion_tmps, '#d62728', 'max')
        annotate_points(ax1, skin_tmps, '#2ca02c', 'max')
    else:
        annotate_points(ax1, lesion_tmps, '#d62728', 'min')
        annotate_points(ax1, skin_tmps, '#2ca02c', 'min')


    
    
    # Enhance plot appearance
    ax1.set_xlabel('Frame Number', fontsize=12)
    ax1.set_ylabel('Temperature (°C)', fontsize=12)
    ax1.tick_params(axis='both', which='major', labelsize=10)
    
    # Add grid with subtle styling
    ax1.grid(True, linestyle=':', alpha=0.4)
    
    # Add legend with larger font
    ax1.legend(loc='best', fontsize=11, framealpha=1)
    
    # Draw reference image using our new drawlocations function
    img2 = drawlocations(video[:,:,0], lesion_locations, stimulation_loc, skin_locations)
    ax2.imshow(img2)
    ax2.axis('off')  # Remove axes for cleaner image display
    
    # Adjust layout and save
    plt.tight_layout(pad=3.0)
    fig.savefig('Contour_Temperature_over_the_time.png', 
                dpi=300, 
                bbox_inches='tight', 
                facecolor='white')
    plt.show()
   

In [173]:
# sampled_video = combined_array.copy()

In [174]:
plot_lesion_temperature(sampled_video,lesion_loc,skin_loc_on_original,plot_type,stimulation_loc)

Frame Number 0: Skin Lesion Temperature 34.4053840637207
Frame Number 0: Skin  Temperature 33.44965362548828
Frame Number 1: Skin Lesion Temperature 36.543739318847656
Frame Number 1: Skin  Temperature 36.53003692626953
Frame Number 2: Skin Lesion Temperature 36.43830490112305
Frame Number 2: Skin  Temperature 36.41215896606445
Frame Number 3: Skin Lesion Temperature 36.4863395690918
Frame Number 3: Skin  Temperature 36.46369934082031
Frame Number 4: Skin Lesion Temperature 36.41157913208008
Frame Number 4: Skin  Temperature 36.33505630493164
Frame Number 5: Skin Lesion Temperature 36.364776611328125
Frame Number 5: Skin  Temperature 36.26103210449219
Frame Number 6: Skin Lesion Temperature 36.395694732666016
Frame Number 6: Skin  Temperature 36.28111267089844
Frame Number 7: Skin Lesion Temperature 36.319637298583984
Frame Number 7: Skin  Temperature 36.20006561279297
Frame Number 8: Skin Lesion Temperature 36.34315490722656
Frame Number 8: Skin  Temperature 36.19108200073242
Frame Nu

# Cropped the Theramal Stimulation 

**Idea:** Even though the area beyond the thermal stimulation contains information, it mainly represents healthy skin without thermal stimulation. To analyze how thermal stimulation affects skin condition, we need to extract only the region that is directly covered by the thermal stimulation.

In [ ]:
idx = 111

img = vid[:,:,idx]


img = drawlocations_OPENCV(img,lesion_loc_on_original,stimulation_loc,skin_loc_on_original)

plt.imshow(img)

In [ ]:
# Task : Mark the thermal stimulation area
cropping_coords,_,_ = mark_coordinates_on_image(move_image)


In [ ]:
# Extract the area across the video which is covered by the thermal stimulation
cropped_video = sampled_video[round(cropping_coords[0][1]):round(cropping_coords[1][1]),round(cropping_coords[0][0]):round(cropping_coords[1][0]),1:]

In [ ]:

# Map the lesion & healthy contour location into cropped video
lesion_location_on_cropped_image = [[int(loc[0]-cropping_coords[0][0]),int(loc[1]-cropping_coords[0][1])] for loc in lesion_loc_on_original]


thermal_stimulation_on_cropped_image = [[int(loc[0]-cropping_coords[0][0]),int(loc[1]-cropping_coords[0][1])] for loc in stimulation_loc]

In [125]:
img = drawlocations(cropped_video[:,:,1],lesion_location_on_cropped_image,thermal_stimulation_on_cropped_image)
plt.imshow(img)

In [181]:
original_shape = cropped_video.shape

In [182]:
reshape_vid = cropped_video.reshape(cropped_video.shape[0]*cropped_video.shape[1],cropped_video.shape[2])

reshape_vid.shape

(3916, 133)

In [ ]:

# Calculate the mean of each pixel across frames. 
mean_values = np.mean(reshape_vid,axis=1)   # (Image_Shape, N_FRAMES) --> Image_Shape




(3916,)

In [184]:
mean_img = mean_values.reshape(original_shape[:2]) # Transform Mean Image into the Shape of the image

In [185]:
%matplotlib tk

fig,ax = plt.subplots(1)

# Create a NEW polygon for this figure
polygon_1 = plt.Polygon(
lesion_location_on_cropped_image,
closed=True,
fill=None,          # No fill
edgecolor='black',    # Bright color
linewidth=2,        # Thicker line
linestyle='--',      # Solid line
alpha=1.0           # Fully opaque
)
    

polygon_2 = plt.Polygon(
    thermal_stimulation_on_cropped_image,
    closed=True,
    fill=None,          # No fill
    edgecolor='black',    # Bright color
    linewidth=2,        # Thicker line
    linestyle='--',      # Solid line
    alpha=1.0           # Fully opaque
)


im = ax.imshow(mean_img, cmap='jet')
ax.add_patch(polygon_1)  # Add to current figure
ax.add_patch(polygon_2)
# Add color bar on the right (default)

fig.colorbar(im,ax=ax)


plt.savefig('MeanImageOfAllFrames.png')

# Principal Component Analysis


In [132]:
def computePCA(data):
	pca = PCA()
	pca.fit(data)
	components = pca.components_
	eigenValues = pca.explained_variance_ratio_
	scores = pca.fit_transform(data)
	
	return components, eigenValues, scores

In [186]:
reshape_vid.shape

(3916, 133)

In [187]:
reshape_vid.T.shape

(133, 3916)

In [133]:
norm_video = reshape_vid.T - mean_values

In [134]:
components, eigenValues, scores = computePCA(norm_video)

In [135]:
# ty = 'hot'


In [136]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl


for idx in range(0, components.shape[0]):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 10))
    fig.suptitle(f'Principal Components {idx}| Variance Explained {np.multiply(eigenValues[idx], 100)}')
    
    # Reshape and plot component
    img = components[idx, :].reshape(original_shape[0:2])
    
    # Plot scores on ax1
    ax1.plot(scores[:, idx])
    ax1.grid(True)
    
    # Plot image on ax2
    imgplot = ax2.imshow(img, cmap=mpl.colormaps['jet'])
    
    # Create a NEW polygon for this figure
    polygon_1 = plt.Polygon(
    lesion_location_on_cropped_image,
    closed=True,
    fill=None,          # No fill
    edgecolor='black',    # Bright color
    linewidth=2,        # Thicker line
    linestyle='--',      # Solid line
    alpha=1.0           # Fully opaque
)
    
    polygon_2 = plt.Polygon(
    thermal_stimulation_on_cropped_image,
    closed=True,
    fill=None,          # No fill
    edgecolor='black',    # Bright color
    linewidth=2,        # Thicker line
    linestyle='--',      # Solid line
    alpha=1.0           # Fully opaque
)
    ax2.add_patch(polygon_1)  # Add to current figure
    ax2.add_patch(polygon_2)
    
    # Set axis limits to match image dimensions
    ax2.set_xlim(0, original_shape[1])
    ax2.set_ylim(original_shape[0], 0)  # Inverted y-axis for images
    
    # Add colorbar
    plt.colorbar(imgplot, ax=ax2)
    
    plt.savefig(f'{video_type}_{str(idx)}.png')
    # plt.close()  # Close the figure to free memory (optional)
    
    if idx == 3:
        break

# Video Building using Component

In [ ]:
# Chose the Prinicipal Component That you want to use to rebuild the video here i have chosen the Second Principal Component
Xhat = np.dot(scores[:,:1],components[:1,:])

In [138]:
Xhat += mean_values

In [139]:
Origianal_Video=Xhat.reshape(-1,original_shape[0],original_shape[1])

In [140]:
Origianal_Video.shape

(133, 44, 89)

In [141]:
# plt.imshow(Origianal_Video[0,:,:])

In [143]:
%matplotlib tk

fig,ax = plt.subplots(1)

# Create a NEW polygon for this figure
polygon_1 = plt.Polygon(
lesion_location_on_cropped_image,
closed=True,
fill=None,          # No fill
edgecolor='black',    # Bright color
linewidth=2,        # Thicker line
linestyle='--',      # Solid line
alpha=1.0           # Fully opaque
)
    

polygon_2 = plt.Polygon(
    thermal_stimulation_on_cropped_image,
    closed=True,
    fill=None,          # No fill
    edgecolor='black',    # Bright color
    linewidth=2,        # Thicker line
    linestyle='--',      # Solid line
    alpha=1.0           # Fully opaque
)


im = ax.imshow(Origianal_Video[0,:,:], cmap='jet')
ax.add_patch(polygon_1)  # Add to current figure
ax.add_patch(polygon_2)
# Add color bar on the right (default)

fig.colorbar(im,ax=ax)


plt.savefig('FirstPrincipleComponent.png')